# 1. INTRODUCCIÓN

(...)

# 2. EXTRACCIÓN DE DATOS

In [36]:
import os
import requests
import time
from dotenv import load_dotenv
from datetime import datetime, timedelta
import pandas as pd

load_dotenv()
COINGECKO_API_KEY = os.getenv("COINGECKO_API_KEY")

CRYPTOS = [
    { "name": "Bitcoin", "binance": "BTCUSDT", "coingecko": "bitcoin" },
    { "name": "Ethereum", "binance": "ETHUSDT", "coingecko": "ethereum" },
    { "name": "Solana", "binance": "SOLUSDT", "coingecko": "solana" },
    { "name": "BinanceCoin", "binance": "BNBUSDT", "coingecko": "binancecoin" },
    { "name": "Ripple", "binance": "XRPUSDT", "coingecko": "ripple" },
]

EXTRACTION_CONFIG = {
    "binance": {
        "url": "https://api.binance.com/api/v3/klines",
        "params": {
            "interval": "1h",
            "startTime": int((datetime.now() - timedelta(days=365)).timestamp() * 1000),
            "endTime": int(datetime.now().timestamp() * 1000)
        }
    },
    "coingecko": {
        "url": "https://api.coingecko.com/api/v3/coins/{id}/market_chart",
        "params": {
            "vs_currency": "usd",
            "days": 365,
            "x_cg_demo_api_key": COINGECKO_API_KEY
        }
    }
}

def extract_binance_full_history(symbol, start_time, end_time, interval="1h"):
    """Paginación para obtener más de 1000 velas de Binance"""
    url = EXTRACTION_CONFIG["binance"]["url"]
    all_data = []
    limit = 1000
    current_start = start_time

    while current_start < end_time:
        params = {
            "symbol": symbol,
            "interval": interval,
            "startTime": current_start,
            "endTime": end_time,
            "limit": limit
        }

        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()

        if not data:
            break

        all_data.extend(data)

        # siguiente batch (última vela + 1ms)
        current_start = data[-1][0] + 1

        time.sleep(0.2)  # evitar rate limit

    return all_data


def extract_data(url, params):
    try:
        response = requests.get(url=url, params=params)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print("Ha ocurrido un error: ", e)


dfs_cryptos = {}

for crypto in CRYPTOS:

    crypto_name = crypto["name"]
    crypto_symbol_binance = crypto["binance"]
    crypto_symbol_coingecko = crypto["coingecko"]

    print(f"---- Procesando {crypto_name}... ----\n")

    # ---------------- BINANCE (PAGINADO) ----------------
    data_binance = extract_binance_full_history(
        crypto_symbol_binance,
        EXTRACTION_CONFIG["binance"]["params"]["startTime"],
        EXTRACTION_CONFIG["binance"]["params"]["endTime"]
    )

    df_binance = pd.DataFrame(data_binance, columns=[
        "open_time", "open_price", "high_price", "low_price",
        "close_price", "volume", "close_time",
        "quote_asset_volume", "number_trades",
        "taker_buy_base_asset_volume",
        "taker_buy_quote_asset_volume",
        "unused_field"
    ])

    df_binance = df_binance[
        ["open_time", "open_price", "high_price", "low_price",
         "close_price", "volume", "close_time"]
    ]

    df_binance["open_time"] = pd.to_datetime(df_binance["open_time"], unit="ms")


    # ---------------- COINGECKO ----------------
    coingecko_url = EXTRACTION_CONFIG["coingecko"]["url"].format(
        id=crypto_symbol_coingecko
    )

    data_coingecko = extract_data(coingecko_url, EXTRACTION_CONFIG["coingecko"]["params"])
    data_coingecko = data_coingecko["market_caps"]

    df_coingecko = pd.DataFrame(data_coingecko, columns=["open_time", "market_cap"])
    df_coingecko["open_time"] = pd.to_datetime(df_coingecko["open_time"], unit="ms")

    df_coingecko = df_coingecko.set_index("open_time").sort_index()
    df_coingecko = df_coingecko.resample("1h").ffill()


    # ---------------- ALINEACIÓN ----------------
    start = max(df_binance["open_time"].min(), df_coingecko.index.min())
    end = min(df_binance["open_time"].max(), df_coingecko.index.max())

    df_binance = df_binance[
        (df_binance["open_time"] >= start) &
        (df_binance["open_time"] <= end)
    ]

    df_coingecko = df_coingecko.loc[start:end]

    df_final = df_binance.merge(
        df_coingecko,
        left_on="open_time",
        right_index=True,
        how="left"
    )

    df_final["crypto"] = crypto_name
    df_final = df_final[["crypto"] + [col for col in df_final.columns if col != "crypto"]] # Ordenar para que el nombre de la crypto sea la primera columna
    dfs_cryptos[crypto_name] = df_final
    print(f"---- Obtención de datos de {crypto_name} ✅ ----\n")


print("---- 🧪 DATASETS FINALES 🧪 ----\n")

for name, df_crypto in dfs_cryptos.items():
    print(f"---- Datos finales obtenidos para el DataFrame de: {name} ----\n")
    display(df_crypto.head(10)) 
    print(df_crypto.shape)

---- Procesando Bitcoin... ----

---- Obtención de datos de Bitcoin ✅ ----

---- Procesando Ethereum... ----

---- Obtención de datos de Ethereum ✅ ----

---- Procesando Solana... ----

---- Obtención de datos de Solana ✅ ----

---- Procesando BinanceCoin... ----

---- Obtención de datos de BinanceCoin ✅ ----

---- Procesando Ripple... ----

---- Obtención de datos de Ripple ✅ ----

---- 🧪 DATASETS FINALES 🧪 ----

---- Datos finales obtenidos para el DataFrame de: Bitcoin ----



,crypto,open_time,open_price,high_price,low_price,close_price,volume,close_time,market_cap
4,Bitcoin,2025-04-14 00:00:00,83760.00000000,85129.69000000,83678.00000000,84325.74000000,1860.02544000,1744592399999,1.658923e+12
5,Bitcoin,2025-04-14 01:00:00,84325.73000000,84687.10000000,84048.01000000,84228.00000000,1013.96485000,1744595999999,1.658923e+12
6,Bitcoin,2025-04-14 02:00:00,84228.00000000,85544.30000000,84036.84000000,85140.70000000,3177.66333000,1744599599999,1.658923e+12
7,Bitcoin,2025-04-14 03:00:00,85140.69000000,85166.84000000,84445.49000000,84566.03000000,3093.33239000,1744603199999,1.658923e+12
8,Bitcoin,2025-04-14 04:00:00,84566.04000000,85110.00000000,84415.25000000,84869.31000000,1719.13731000,1744606799999,1.658923e+12
9,Bitcoin,2025-04-14 05:00:00,84869.31000000,84990.57000000,84343.73000000,84575.18000000,599.79720000,1744610399999,1.658923e+12
10,Bitcoin,2025-04-14 06:00:00,84575.19000000,84644.91000000,84279.81000000,84624.21000000,617.23501000,1744613999999,1.658923e+12
11,Bitcoin,2025-04-14 07:00:00,84624.21000000,84787.07000000,84211.06000000,84580.76000000,690.45295000,1744617599999,1.658923e+12
12,Bitcoin,2025-04-14 08:00:00,84580.77000000,84957.27000000,84440.00000000,84763.18000000,771.96623000,1744621199999,1.658923e+12
13,Bitcoin,2025-04-14 09:00:00,84763.18000000,84912.94000000,84250.00000000,84442.72000000,632.50132000,1744624799999,1.658923e+12


(8756, 9)
---- Datos finales obtenidos para el DataFrame de: Ethereum ----



,crypto,open_time,open_price,high_price,low_price,close_price,volume,close_time,market_cap
4,Ethereum,2025-04-14 00:00:00,1597.76000000,1633.98000000,1595.56000000,1611.85000000,37744.64520000,1744592399999,1.923826e+11
5,Ethereum,2025-04-14 01:00:00,1611.85000000,1622.58000000,1609.09000000,1617.34000000,13380.27130000,1744595999999,1.923826e+11
6,Ethereum,2025-04-14 02:00:00,1617.34000000,1660.32000000,1614.43000000,1635.44000000,54996.73260000,1744599599999,1.923826e+11
7,Ethereum,2025-04-14 03:00:00,1635.44000000,1637.68000000,1625.85000000,1629.59000000,20711.70970000,1744603199999,1.923826e+11
8,Ethereum,2025-04-14 04:00:00,1629.60000000,1642.19000000,1627.67000000,1636.52000000,16817.90830000,1744606799999,1.923826e+11
9,Ethereum,2025-04-14 05:00:00,1636.53000000,1640.16000000,1620.65000000,1624.38000000,18804.05050000,1744610399999,1.923826e+11
10,Ethereum,2025-04-14 06:00:00,1624.38000000,1628.04000000,1614.56000000,1623.77000000,15259.59540000,1744613999999,1.923826e+11
11,Ethereum,2025-04-14 07:00:00,1623.76000000,1636.40000000,1612.63000000,1635.57000000,18010.92070000,1744617599999,1.923826e+11
12,Ethereum,2025-04-14 08:00:00,1635.58000000,1642.12000000,1629.34000000,1638.28000000,13917.14100000,1744621199999,1.923826e+11
13,Ethereum,2025-04-14 09:00:00,1638.27000000,1645.24000000,1631.24000000,1641.34000000,15176.71370000,1744624799999,1.923826e+11


(8756, 9)
---- Datos finales obtenidos para el DataFrame de: Solana ----



...

(8756, 9)
---- Datos finales obtenidos para el DataFrame de: BinanceCoin ----



,crypto,open_time,open_price,high_price,low_price,close_price,volume,close_time,market_cap
4,BinanceCoin,2025-04-14 00:00:00,584.29000000,587.42000000,583.80000000,585.99000000,8337.50300000,1744592399999,8.513484e+10
5,BinanceCoin,2025-04-14 01:00:00,586.00000000,587.25000000,585.65000000,586.20000000,5591.49400000,1744595999999,8.513484e+10
6,BinanceCoin,2025-04-14 02:00:00,586.20000000,590.86000000,586.19000000,589.61000000,11121.73300000,1744599599999,8.513484e+10
7,BinanceCoin,2025-04-14 03:00:00,589.61000000,592.00000000,587.78000000,588.20000000,12638.71600000,1744603199999,8.513484e+10
8,BinanceCoin,2025-04-14 04:00:00,588.20000000,590.39000000,587.96000000,590.00000000,4275.28700000,1744606799999,8.513484e+10
9,BinanceCoin,2025-04-14 05:00:00,590.01000000,590.40000000,587.05000000,587.77000000,4563.68300000,1744610399999,8.513484e+10
10,BinanceCoin,2025-04-14 06:00:00,587.76000000,589.52000000,587.16000000,589.10000000,9856.14800000,1744613999999,8.513484e+10
11,BinanceCoin,2025-04-14 07:00:00,589.10000000,590.87000000,587.53000000,590.76000000,6434.39500000,1744617599999,8.513484e+10
12,BinanceCoin,2025-04-14 08:00:00,590.76000000,591.50000000,589.74000000,590.73000000,5751.95700000,1744621199999,8.513484e+10
13,BinanceCoin,2025-04-14 09:00:00,590.72000000,591.72000000,588.48000000,589.09000000,5267.72800000,1744624799999,8.513484e+10


(8756, 9)
---- Datos finales obtenidos para el DataFrame de: Ripple ----



,crypto,open_time,open_price,high_price,low_price,close_price,volume,close_time,market_cap
4,Ripple,2025-04-14 00:00:00,2.11960000,2.15280000,2.11240000,2.13870000,7271895.00000000,1744592399999,1.234939e+11
5,Ripple,2025-04-14 01:00:00,2.13870000,2.15080000,2.12810000,2.13450000,5098767.00000000,1744595999999,1.234939e+11
6,Ripple,2025-04-14 02:00:00,2.13460000,2.18040000,2.12770000,2.15280000,8027087.00000000,1744599599999,1.234939e+11
7,Ripple,2025-04-14 03:00:00,2.15270000,2.15500000,2.13650000,2.14000000,5574696.00000000,1744603199999,1.234939e+11
8,Ripple,2025-04-14 04:00:00,2.14010000,2.15940000,2.13430000,2.15340000,4280753.00000000,1744606799999,1.234939e+11
9,Ripple,2025-04-14 05:00:00,2.15340000,2.15920000,2.12930000,2.13380000,5930558.00000000,1744610399999,1.234939e+11
10,Ripple,2025-04-14 06:00:00,2.13370000,2.13660000,2.11820000,2.11820000,6981503.00000000,1744613999999,1.234939e+11
11,Ripple,2025-04-14 07:00:00,2.11820000,2.12970000,2.10380000,2.12810000,6381520.00000000,1744617599999,1.234939e+11
12,Ripple,2025-04-14 08:00:00,2.12820000,2.14320000,2.12270000,2.13780000,4709968.00000000,1744621199999,1.234939e+11
13,Ripple,2025-04-14 09:00:00,2.13770000,2.14490000,2.11680000,2.12730000,4830809.00000000,1744624799999,1.234939e+11


(8756, 9)


# 3. ANÁLISIS EXPLORATORIO Y DIAGNÓSTICO DE LA CALIDAD DE LOS DATOS

In [37]:
# Para cada DataFrame, realizar un diagnóstico para identificar problemas de calidad, como tipos de datos incorrectos, valores nulos, etc.

def quality_check(df, df_name):
    """ Función para dignosticar la estructura y la calidad de los datos provenientes de cada DataFrame """
    
    print(f"\n{'='*60}")
    print(f"DIAGNÓSTICO DEL DATAFRAME: {df_name}")
    print(f"{'='*60}")
    
    # Información general
    print("\n--- INFO GENERAL ---")
    df.info()
    
    # Conteo de nulos reales
    print("\n--- NULOS REALES (isnull) ---")
    print(df.isnull().sum())
    
    print("\n--- % DE NULOS ---")
    print((df.isnull().mean() * 100).round(2))
    
    # Valores sospechosos típicos
    print("\n--- VALORES SOSPECHOSOS ---")
    valores_sospechosos = ["-", "null", "NULL", "None", "", " "]
    
    for val in valores_sospechosos:
        conteo = (df == val).sum().sum()
        if conteo > 0:
            print(f"Valor '{val}' encontrado {conteo} veces")
    
    # Detección de tipos incorrectos
    print("\n--- COLUMNAS OBJECT (posibles tipos incorrectos) ---")
    print(df.select_dtypes(include=["object", "string"]).columns.tolist())

for name, df_crypto in dfs_cryptos.items():
    quality_check(df_crypto, name)


DIAGNÓSTICO DEL DATAFRAME: Bitcoin

--- INFO GENERAL ---
<class 'pandas.DataFrame'>
RangeIndex: 8756 entries, 4 to 8759
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   crypto       8756 non-null   str           
 1   open_time    8756 non-null   datetime64[ms]
 2   open_price   8756 non-null   str           
 3   high_price   8756 non-null   str           
 4   low_price    8756 non-null   str           
 5   close_price  8756 non-null   str           
 6   volume       8756 non-null   str           
 7   close_time   8756 non-null   int64         
 8   market_cap   8756 non-null   float64       
dtypes: datetime64[ms](1), float64(1), int64(1), str(6)
memory usage: 615.8 KB

--- NULOS REALES (isnull) ---
crypto         0
open_time      0
open_price     0
high_price     0
low_price      0
close_price    0
volume         0
close_time     0
market_cap     0
dtype: int64

--- % DE NULOS ---
crypto      

Con los resultados obtenidos, se observa que no existen valores nulos en el dataset, lo que indica que los datos extraídos de las APIs presentan un buen nivel de consistencia.

En cuanto a la conversión de tipos de datos, es necesario realizar los siguientes ajustes:

* Las variables *open_price*, *high_price*, *low_price*, *close_price* y *volume* deben convertirse a tipo float64, con el objetivo de preservar los valores decimales y mantener la máxima precisión en los cálculos posteriores.
* La variable *close_time* debe convertirse a tipo datetime, lo que permitirá un manejo adecuado de las fechas y facilitará su uso en análisis temporales.

# 4. TRANSFORMACIÓN Y LIMPIEZA DE LOS DATOS

In [38]:
# Transform:
## Conversión de tipos

numeric_cols = ["open_price", "high_price", "low_price", "close_price", "volume"]
dfs_correct_types = {}

def convert_dtypes(df, df_name):
    """ Función para convertir los tipos correctamente en cada DataFrame """
    
    print(f"\n{'='*60}")
    print(f"CONVERSIÓN DE TIPOS DEL DATAFRAME: {df_name}")
    print(f"{'='*60}")

    if "close_time" in df.columns:
        df["close_time"] = pd.to_datetime(df["close_time"], errors="coerce", unit="ms")

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype(float)

    # Revisamos cómo está ahora cada DataFrame
    print("\n--- INFO GENERAL POSTERIOR ---")
    df.info()

    dfs_correct_types[df_name] = df

for name, df_crypto in dfs_cryptos.items():
    convert_dtypes(df_crypto, name)

print("\nFINALIZADA CON ÉXITO LA CONVERSIÓN DE TIPOS ✅")
print(f"\n🧪 DataFrames con tipos correctos disponibles en 'dfs_correct_types': {list(dfs_correct_types.keys())}")



CONVERSIÓN DE TIPOS DEL DATAFRAME: Bitcoin

--- INFO GENERAL POSTERIOR ---
<class 'pandas.DataFrame'>
RangeIndex: 8756 entries, 4 to 8759
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   crypto       8756 non-null   str           
 1   open_time    8756 non-null   datetime64[ms]
 2   open_price   8756 non-null   float64       
 3   high_price   8756 non-null   float64       
 4   low_price    8756 non-null   float64       
 5   close_price  8756 non-null   float64       
 6   volume       8756 non-null   float64       
 7   close_time   8756 non-null   datetime64[ms]
 8   market_cap   8756 non-null   float64       
dtypes: datetime64[ms](2), float64(6), str(1)
memory usage: 615.8 KB

CONVERSIÓN DE TIPOS DEL DATAFRAME: Ethereum

--- INFO GENERAL POSTERIOR ---
<class 'pandas.DataFrame'>
RangeIndex: 8756 entries, 4 to 8759
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype         


# 5. ALMACENAMIENTO DE LOS DATOS PARA EL DATA LAKE

In [39]:
# Load:
## Guardar todos los DataFrames obtenidos, incluido uno final con TODOS los datos de las CRYPTOS

output_dir = "../data/processed"
os.makedirs(output_dir, exist_ok=True)

# DataFrame final que contiene los datos de TODAS las cryptos
df_final = pd.concat(dfs_correct_types.values(), ignore_index=True)
print("\n--- INFO GENERAL DEL DATAFRAME CON TODAS LAS COINS ---")
df_final.info()

def load_data(df, df_name):
    """ Función para almacenar los datos en un Data Lake """

    print(f"\n{'='*60}")
    print(f"ALMACENANDO EL DATAFRAME DE: {df_name}")
    print(f"{'='*60}")

    df.to_parquet(
        f"{output_dir}/{df_name.lower()}.parquet",
        index=False,
        engine="fastparquet"
    )

    print(f"\nDatos de {df_name} almacenados correctamente ✅")


for name, df_crypto in dfs_correct_types.items():
    load_data(df_crypto, name)

load_data(df_final, "all_cryptos")
print("\nFINALIZADA CON ÉXITO LA CREACIÓN DEL DATA LAKE ✅")


--- INFO GENERAL DEL DATAFRAME CON TODAS LAS COINS ---
<class 'pandas.DataFrame'>
RangeIndex: 43780 entries, 0 to 43779
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   crypto       43780 non-null  str           
 1   open_time    43780 non-null  datetime64[ms]
 2   open_price   43780 non-null  float64       
 3   high_price   43780 non-null  float64       
 4   low_price    43780 non-null  float64       
 5   close_price  43780 non-null  float64       
 6   volume       43780 non-null  float64       
 7   close_time   43780 non-null  datetime64[ms]
 8   market_cap   43780 non-null  float64       
dtypes: datetime64[ms](2), float64(6), str(1)
memory usage: 3.0 MB

ALMACENANDO EL DATAFRAME DE: Bitcoin

Datos de Bitcoin almacenados correctamente ✅

ALMACENANDO EL DATAFRAME DE: Ethereum

Datos de Ethereum almacenados correctamente ✅

ALMACENANDO EL DATAFRAME DE: Solana

Datos de Solana almacenados corre